# Qwen-Image-2.1 图片编辑 → 自动上传 GitHub（Colab 免费 GPU）
**仅图像编辑｜不用 LoRA｜主模型 GGUF Q4｜结果同步到 \`mjw3177109/qwen2.1tu/images/\`**

⚠️ 这个仓库是公开的，编辑后的 PNG 会公开。不得将自己的 GitHub Token 放在 Notebook 代码或输出里。

### 第一次使用
1. 在 [GitHub Fine-grained tokens](https://github.com/settings/personal-access-tokens) 创建仅对 \`mjw3177109/qwen2.1tu\` 有效、**Contents: Read and write** 的 Token。
2. Colab 左边栏点击 **🔑 Secrets**，创建 \`GITHUB_TOKEN\` 并粘贴 Token，允许 Notebook 访问。
3. 运行时 → 更改运行时类型 → GPU（若免费配额可用），然后依次执行各代码格。
4. 上传原图，修改下面的 \`EDIT_PROMPT\`，运行最后的编辑格：**生成成功后立即自动上传 GitHub**。

这是未经免费 T4 端到端验证的实验 Notebook。单 T4 16GB 显存以及系统 RAM 可能无法运行这套 11GB 量化模型。

In [ ]:
# 01 / 配置 + GPU、磁盘 + GitHub Token 预检查（先执行，避免白下载模型）
from pathlib import Path
import os, sys, shutil, subprocess, time, io, json, re, uuid, base64
import requests

REPO = "mjw3177109/qwen2.1tu"
BRANCH = "main"
EDIT_PROMPT = "<image1> 保持人物身份、姿态、背景与二维动漫风格不变，只把衣服变成深蓝色。"
EDIT_RESOLUTION = 512       # 先用 512；成功后可尝试 640 或 768
ENCODER_MODE = "w4a8"      # 小体积 6.31GB；不兼容可试 int8（更占磁盘）
SEED = 42
ROOT = Path("/content/qwen21_light")
COMFY = ROOT / "ComfyUI"
OUT = Path("/content/qwen21_outputs")
OUT.mkdir(parents=True, exist_ok=True)
(ROOT / "tmp").mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(ROOT / "hf_cache")
os.environ["HF_HUB_CACHE"] = str(ROOT / "hf_cache/hub")
os.environ["HF_XET_CACHE"] = str(ROOT / "hf_cache/xet")
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["PIP_NO_CACHE_DIR"] = "1"
os.environ["TMPDIR"] = str(ROOT / "tmp")
os.environ["XDG_CACHE_HOME"] = str(ROOT / "xdg_cache")

from google.colab import userdata
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise RuntimeError("请先在 Colab 左侧 🔑 Secrets 添加 GITHUB_TOKEN 并允许 Notebook 访问。")

GH_HEADERS = {
    "Authorization": "Bearer " + GITHUB_TOKEN,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
probe = requests.get("https://api.github.com/repos/" + REPO, headers=GH_HEADERS, timeout=25)
if probe.status_code != 200:
    raise RuntimeError("GitHub 验证失败：HTTP " + str(probe.status_code) + "。检查 Token 是否授权此仓库。")
print("GitHub 仓库连接成功:", REPO)
if not probe.json().get("permissions", {}).get("push", False):
    print("⚠️ Token 未被确认具有 push 权限；请确保仓库 Contents 是 Read and write。")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU，请在运行时设置里选择 GPU。Colab 免费额度可能暂不可用。")
for i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(i)
    print("GPU:", prop.name, f"{prop.total_memory/2**30:.1f} GiB")
free = shutil.disk_usage("/content").free
print("Colab 临时磁盘剩余:", round(free/2**30, 1), "GiB")
if free < 17 * 2**30:
    raise RuntimeError("临时磁盘不足 17 GiB，停止以避免下载失败。")
try:
    import psutil
    print("系统总 RAM:", round(psutil.virtual_memory().total/2**30, 1), "GiB")
except ImportError:
    pass
assert ENCODER_MODE in ("w4a8", "int8")

In [ ]:
# 02 / 安装 ComfyUI + GGUF 扩展（复用 Colab 自带 torch）
def run_cmd(args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=str(cwd) if cwd else None, check=True)

if not (COMFY / "main.py").exists():
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", COMFY])
node_dir = COMFY / "custom_nodes" / "ComfyUI-GGUF"
if not (node_dir / "nodes.py").exists():
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/city96/ComfyUI-GGUF.git", node_dir])

requirements = []
for line in (COMFY / "requirements.txt").read_text(encoding="utf-8").splitlines():
    s = line.strip()
    if s and not s.startswith("#") and not re.match(r"^(torch|torchvision|torchaudio)(\b|[<>=~!\[])", s, flags=re.I):
        requirements.append(line)
req = ROOT / "requirements_without_torch.txt"
req.write_text("\n".join(requirements) + "\n", encoding="utf-8")
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", req])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "huggingface_hub>=0.33", "gguf", "requests"])
gguf_req = node_dir / "requirements.txt"
if gguf_req.exists():
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", gguf_req])
print("ComfyUI 环境安装完成。")

In [ ]:
# 03 / 仅下载图片编辑需要的 3 个权重（模型存 Colab 临时磁盘，不保存到 GitHub）
from huggingface_hub import hf_hub_download

MODELS = COMFY / "models"
GGUF_FILE = "qwen-image-2.1-turbo-Q4_K_M.gguf"
ENC_FILE = ("qwen3vl_8b_w4a8.safetensors" if ENCODER_MODE == "w4a8"
            else "qwen3vl_8b_int8_convrot.safetensors")
VAE_FILE = "qwen_image_2.1_vae_bf16.safetensors"

weights = [
    ("abenzerps/Qwen-Image-2.1-Turbo-Quantized", GGUF_FILE, MODELS/"diffusion_models", 4.05),
    ("Comfy-Org/Qwen-Image-2.1", "text_encoders/"+ENC_FILE, MODELS, 6.31 if ENCODER_MODE=="w4a8" else 8.71),
    ("Comfy-Org/Qwen-Image-2.1", "vae/"+VAE_FILE, MODELS, 0.65),
]
for repo_id, filename, folder, size_gb in weights:
    folder.mkdir(parents=True, exist_ok=True)
    target = folder/filename
    if target.is_file() and target.stat().st_size > size_gb * 1e9 * 0.90:
        print("已存在，跳过:", target.name)
        continue
    if shutil.disk_usage("/content").free < size_gb*1e9 + 2e9:
        raise RuntimeError("磁盘不足，未下载 " + filename)
    print("下载:", repo_id, filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(folder))
    if not target.is_file():
        raise FileNotFoundError("下载后找不到模型: " + str(target))
    print("完成:", target.name, round(target.stat().st_size/1e9, 2), "GB")
print("三个必需模型下载完成。")

In [ ]:
# 04 / 启动本地 ComfyUI 并检查所需节点
LOG = ROOT / "comfyui.log"
URL = "http://127.0.0.1:8188"
def healthy():
    try:
        return requests.get(URL+"/system_stats", timeout=3).ok
    except requests.RequestException:
        return False

if not healthy():
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--lowvram", "--disable-auto-launch", "--output-directory", str(OUT),
           "--temp-directory", str(ROOT/"tmp"), "--input-directory", str(COMFY/"input")]
    with LOG.open("a", encoding="utf-8") as f:
        comfy_process = subprocess.Popen(cmd, cwd=COMFY, stdout=f, stderr=subprocess.STDOUT)
    for _ in range(90):
        if healthy():
            break
        if comfy_process.poll() is not None:
            break
        time.sleep(2)
if not healthy():
    print("\n".join(LOG.read_text(errors="replace").splitlines()[-90:]))
    raise RuntimeError("ComfyUI 启动失败")

infos = requests.get(URL+"/object_info", timeout=45).json()
needed = ("UnetLoaderGGUF", "CLIPLoader", "VAELoader", "LoadImage",
          "TextEncodeQwenImage21", "KSamplerSelect", "ManualSigmas",
          "SamplerCustom", "VAEDecode", "SaveImage")
missing = [node for node in needed if node not in infos]
if missing:
    raise RuntimeError("当前 ComfyUI 版本缺少节点: " + ", ".join(missing))
print("编辑节点全部存在，ComfyUI 已就绪。")

In [ ]:
# 05 / 从电脑上传原图，自动缩小像素量并保留长宽比
from google.colab import files as colab_files
from PIL import Image, ImageOps
from IPython.display import display

uploaded = colab_files.upload()
if not uploaded:
    raise RuntimeError("没有上传图片。")
name, raw = next(iter(uploaded.items()))
if Path(name).suffix.lower() not in (".png", ".jpg", ".jpeg", ".webp"):
    raise ValueError("只允许 PNG/JPG/WEBP")
im = ImageOps.exif_transpose(Image.open(io.BytesIO(raw))).convert("RGB")
limit = EDIT_RESOLUTION**2
if im.width*im.height > limit:
    scale = (limit/(im.width*im.height))**0.5
    w = max(32, round(im.width*scale/32)*32)
    h = max(32, round(im.height*scale/32)*32)
    im = im.resize((w,h), Image.Resampling.LANCZOS)
(COMFY/"input").mkdir(parents=True,exist_ok=True)
INPUT_NAME = "qwen21_edit_input.png"
im.save(COMFY/"input"/INPUT_NAME)
print("上传用于编辑的图片:", name, "模型尺寸:", im.size)
display(im)

In [ ]:
# 06 / 图像编辑工作流（参考图确实接入 Qwen-Image-2.1 节点）
import secrets
SIGMAS = "1.0, 0.978453, 0.95418, 0.926626, 0.89508, 0.845148, 0.704534, 0.414568, 0.0"

def make_graph(prompt, input_name, seed):
    return {
        "1": {"class_type":"UnetLoaderGGUF","inputs":{"unet_name":GGUF_FILE}},
        "2": {"class_type":"CLIPLoader","inputs":{"clip_name":ENC_FILE, "type":"qwen_image", "device":"default"}},
        "3": {"class_type":"VAELoader","inputs":{"vae_name":VAE_FILE}},
        "4": {"class_type":"LoadImage","inputs":{"image":input_name}},
        "5": {"class_type":"TextEncodeQwenImage21","inputs":{
            "clip":["2",0], "vae":["3",0], "prompt":prompt, "negative_prompt":"",
            "resolution":EDIT_RESOLUTION, "images.image_1":["4",0]}},
        "6": {"class_type":"KSamplerSelect","inputs":{"sampler_name":"euler"}},
        "7": {"class_type":"ManualSigmas","inputs":{"sigmas":SIGMAS}},
        "8": {"class_type":"SamplerCustom","inputs":{
            "model":["1",0], "add_noise":True, "noise_seed":int(seed),
            "cfg":1.0, "positive":["5",0], "negative":["5",1],
            "sampler":["6",0], "sigmas":["7",0], "latent_image":["5",2]}},
        "9": {"class_type":"VAEDecode","inputs":{"samples":["8",0], "vae":["3",0]}},
        "10":{"class_type":"SaveImage","inputs":{
            "images":["9",0], "filename_prefix":"Qwen21_Edit_Github"}}
    }

# 校验参考图连接，防止误生成无关图片
assert make_graph("test", INPUT_NAME, SEED)["5"]["inputs"]["images.image_1"] == ["4",0]
print("修改提示词:", EDIT_PROMPT)

In [ ]:
# 07 / 编辑图片 + 立即上传 GitHub images/（自动化主格）
from datetime import datetime, timezone

def publish_png(path: Path):
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size >= 95_000_000:
        raise RuntimeError("图片太大，不适合 GitHub Contents API；请先压缩图片。")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    remote_path = f"images/{stamp}_{uuid.uuid4().hex[:10]}.png"
    endpoint = f"https://api.github.com/repos/{REPO}/contents/{remote_path}"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    response = requests.put(endpoint, headers=GH_HEADERS, json={
        "message": "images: add Qwen-Image-2.1 edited PNG",
        "content": encoded,
        "branch": BRANCH
    }, timeout=120)
    if response.status_code not in (200, 201):
        # 避免把请求 Header、凭证打印出去
        raise RuntimeError("GitHub 上传失败 HTTP " + str(response.status_code) +
                           "，响应：" + response.text[:600])
    uploaded_sha = response.json().get("content", {}).get("sha")
    check = requests.get(endpoint, headers=GH_HEADERS, params={"ref":BRANCH}, timeout=25)
    if check.status_code != 200 or check.json().get("sha") != uploaded_sha:
        raise RuntimeError("GitHub 已报告提交，但回读 SHA 验证失败。请检查仓库。")
    url = "https://github.com/" + REPO + "/blob/" + BRANCH + "/" + remote_path
    print("✅ PNG 已上传 GitHub 并验证:", url)
    return url

def edit_and_publish(prompt=EDIT_PROMPT, seed=SEED):
    payload = {"prompt":make_graph(prompt, INPUT_NAME, seed),
               "client_id":secrets.token_hex(12)}
    res = requests.post(URL+"/prompt", json=payload, timeout=90)
    if not res.ok:
        print("ComfyUI 节点验证返回:", res.text[:2000])
    res.raise_for_status()
    response = res.json()
    if response.get("node_errors"):
        raise RuntimeError("节点错误：" + repr(response["node_errors"])[:2000])
    task_id = response["prompt_id"]
    print("图片编辑已提交：", task_id)
    start = time.monotonic()
    while time.monotonic()-start < 3600:
        history = requests.get(URL+"/history/"+task_id, timeout=25).json()
        record = history.get(task_id)
        if record:
            status = record.get("status", {})
            if not status.get("completed", False):
                print("\n".join(LOG.read_text(errors="replace").splitlines()[-70:]))
                raise RuntimeError("图片编辑失败：" + repr(status)[:1500])
            files = record.get("outputs",{}).get("10",{}).get("images",[])
            if not files:
                raise RuntimeError("任务已完成但没有生成 PNG")
            urls = []
            for entry in files:
                path = OUT/entry.get("subfolder","")/entry["filename"]
                if not path.is_file():
                    raise FileNotFoundError(path)
                display(Image.open(path))
                print("本地文件:", path)
                urls.append(publish_png(path))
            return urls
        time.sleep(3)
    raise TimeoutError("编辑超时，请检查运行日志")

GITHUB_IMAGE_URLS = edit_and_publish()
print("完成，上传了", len(GITHUB_IMAGE_URLS), "张图片。")

In [ ]:
# 08 / 可选：下载本次编辑的 PNG 到你的电脑（不会重新生成）
if "GITHUB_IMAGE_URLS" not in globals():
    raise RuntimeError("请先运行 07 生成图片并上传 GitHub。")
import zipfile
zip_path = Path("/content/qwen21_edited_results.zip")
png_files = sorted(OUT.rglob("*.png"))
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in png_files:
        zf.write(f, arcname=f.relative_to(OUT))
print("下载图片压缩包（不包含模型）:", zip_path)
colab_files.download(str(zip_path))

## 出错排查
- **没有 GPU**：在 Colab「运行时」切换到 GPU。免费版不保证随时可用。
- **缺少 GITHUB_TOKEN**：左侧 🔑 Secrets 新增 \`GITHUB_TOKEN\`，确保允许 Notebook 访问，不要把 Token 粘进代码。
- **HTTP 403 上传失败**：Token 要对目标仓库授予 Contents: Read and write 权限。
- **模型下载失败/内存不足**：清理临时盘、重开会话，或改用 Kaggle / 更高内存运行环境。
- **Qwen3-VL W4A8 失败**：将 \`ENCODER_MODE\` 改成 \`int8\` 并从头运行（下载更大）。
- **节点验证失败**：检查 ComfyUI 的版本与 \`comfyui.log\`，不要把未经证实的输出当成功。
- **上传后找不到文件**：查看 <https://github.com/mjw3177109/qwen2.1tu/tree/main/images>。
- **仓库是公开的**：任何人都能查看和下载上传的编辑图片。
